# Quickstart

In [1]:
from pathlib import Path
from metasmith.python_api import *
from metasmith import examples

from local.constants import WORKSPACE_ROOT

In [4]:
EXAMPLES_DIR = WORKSPACE_ROOT/"src/metasmith/example_resources"
dtypes = DataTypeLibrary.Load(EXAMPLES_DIR/"types/minimal_genomics.yml")

xgdb_path = EXAMPLES_DIR/"data/fosmid.xgdb"
xgdb = DataInstanceLibrary.Load(xgdb_path)
# xgdb = DataInstanceLibrary(xgdb_path)
# xgdb.AddTypeLibrary("genomics", dtypes)
# xgdb.Add([
#     (EXAMPLES_DIR/"fosmid.fna", "./fosmid.fna", "genomics::contigs"),
# ])
# xgdb.Save()

refdb_path = EXAMPLES_DIR/"data/references.xgdb"
refdb = DataInstanceLibrary.Load(refdb_path)
# refdb = DataInstanceLibrary(refdb_path)
# refdb.AddTypeLibrary("genomics", dtypes)
# refdb.Add([
#     (EXAMPLES_DIR/"swissprot_bcaa.fna", "./swissprot_bcaa.fna", "genomics::aa_sequences"),
# ])
# refdb.Save()

trans_path = EXAMPLES_DIR/"transforms/gene_annotation.xgdb"
transforms = TransformInstanceLibrary.Load(trans_path)
# transforms = TransformInstanceLibrary(trans_path)
# transforms.AddTypeLibrary("genomics", dtypes)
# transforms.AddStub("pprodigal")
# transforms.AddStub("diamond")
# transforms.AddStub("make_diamond_db")
# transforms.Save()

In [ ]:
agent = Agent(
    home = Source.FromLocal(Path("./metasmith_home").resolve()),
)
# agent.Deploy()

2025-03-18_18-31-21  | /home/tony/workspace/tools/Metasmith/docs/source/metasmith_home
2025-03-18_18-31-21  | /home/tony
2025-03-18_18-31-21  | >>> AGENT_HOME=/home/tony/workspace/tools/Metasmith/docs/source/metasmith_home
2025-03-18_18-31-21  | >>> mkdir -p $AGENT_HOME
2025-03-18_18-31-21  | >>> mkdir -p /home/tony/.globus
2025-03-18_18-31-21  | >>> mkdir -p /home/tony/.globusonline
2025-03-18_18-31-21  | >>> apptainer pull/metasmith.sif docker://quay.io/hallamlab/metasmith:latest
2025-03-18_18-31-21  | staged [msm_stub]
2025-03-18_18-31-21  | staged [msm]
2025-03-18_18-31-21  | >>> cd /home/tony/workspace/tools/Metasmith/docs/source/metasmith_home && ./msm api deploy_from_container
2025-03-18_18-31-22  | 2025-03-18_18-31-22  | api call to [deploy_from_container] with [{}]
2025-03-18_18-31-22  | 2025-03-18_18-31-22  | deploying to [/ws]
2025-03-18_18-31-22  | 2025-03-18_18-31-22  | deploying relay server to [/ws/relay/msm_relay]
2025-03-18_18-31-22  | 2025-03-18_18-31-22  | deployment

In [8]:
task = agent.GenerateWorkflow(
    given=[xgdb, refdb],
    transforms=[transforms],
    targets=[
        dtypes["orf_annotations"].WithLineage([dtypes["contigs"]]),
    ]
)
task.plan.steps

[WorkflowStep(order=1, uses=[DataInstance(path=PosixPath('fosmid.fna'), dtype=<{data:DNA sequence,format:FASTA}:4M4PqXwA>, dtype_name='genomics::contigs', parent_lib=<metasmith.models.libraries.DataInstanceLibrary object at 0x7f3144132cf0>), DataInstance(path=PosixPath('pprodigal.oci.uri'), dtype=<{data:software container,format:OCI,provides:pprodigal}:90LdbjQO>, dtype_name='genomics::oci_image_pprodigal', parent_lib=<metasmith.models.libraries.DataInstanceLibrary object at 0x7f30a6b82450>)], produces=[DataInstance(path=PosixPath('orfs.faa'), dtype=(D:{"data":"Amino acid sequence"}-{"format":"FASTA"}), dtype_name='genomics::aa_sequences', parent_lib=<metasmith.models.libraries.TransformInstanceLibrary object at 0x7f31454f05f0>)], transform=TransformInstance(protocol=<function protocol at 0x7f3097fd8180>, model={{"data":"DNA sequence"}-{"format":"FASTA"}},{{"data":"software container"}-{"format":"OCI"}-{"provides":"pprodigal"}}->{{"data":"Amino acid sequence"}-{"format":"FASTA"}}, outpu

In [9]:
agent.StageWorkflow(task)

2025-03-18_18-36-57  | connecting to deployed agent
2025-03-18_18-36-57  | starting relay service
 | > 2025-03-18_18-36-58  | relay server started with pid: [150150]
2025-03-18_18-36-59  | sending metadata for workflow [qLJCf77e]
2025-03-18_18-37-00  | staging
 | > 2025-03-18_18-37-01  | api call to [stage_workflow] with [{'task_key': 'qLJCf77e'}]


E| > Traceback (most recent call last):
E| >   File "/opt/conda/envs/metasmith_env/bin/metasmith", line 8, in <module>
E| >     sys.exit(main())
E| >              ^^^^^^
E| >   File "/opt/conda/envs/metasmith_env/lib/python3.12/site-packages/metasmith/coms/cli.py", line 92, in main
E| >     COMMANDS.get(# calls command function with args
E| >   File "/opt/conda/envs/metasmith_env/lib/python3.12/site-packages/metasmith/coms/cli.py", line 67, in api
E| >     HandleRequest(args.endpoint, body)
E| >   File "/opt/conda/envs/metasmith_env/lib/python3.12/site-packages/metasmith/coms/api.py", line 51, in HandleRequest
E| >     _ENDPOINTS[endpoint](api, body)
E| >   File "/opt/conda/envs/metasmith_env/lib/python3.12/site-packages/metasmith/coms/api.py", line 22, in stage_workflow
E| >     StageWorkflow(task_key)
E| >   File "/opt/conda/envs/metasmith_env/lib/python3.12/site-packages/metasmith/agents.py", line 426, in StageWorkflow
E| >     task = WorkflowTask.Load(task_path)
E| >            ^^^

2025-03-18_18-37-02  | closing connection


In [ ]:
agent.